# Experiment Tracking & Reproducibility — Notebook

**Week 10 · CS 203 · Software Tools and Techniques for AI**

Prof. Nipun Batra · IIT Gandhinagar

This notebook covers:
1. Grid Search vs Random Search comparison
2. Bayesian optimization with Optuna
3. PyTorch reproducibility
4. W&B experiment tracking

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install scikit-learn numpy matplotlib optuna torch wandb

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV,
    StratifiedKFold
)
from scipy.stats import randint, uniform
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Generate Dataset

Synthetic movie success dataset with 8 features.

In [ ]:
np.random.seed(42)
n_samples = 1000
X = np.random.rand(n_samples, 8)
y = (X[:, 0] * 2 + X[:, 2] - X[:, 5] + np.random.randn(n_samples) * 0.3 > 1).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Class balance: {y_train.mean():.2%} positive")

## 2. Grid Search

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5, 10],
}

n_combos = 1
for v in param_grid.values():
    n_combos *= len(v)
print(f"Grid search: {n_combos} combinations × 5 folds = {n_combos * 5} fits")

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
)

start = time.time()
grid.fit(X_train, y_train)
grid_time = time.time() - start

print(f"\nBest score: {grid.best_score_:.4f}")
print(f"Best params: {grid.best_params_}")
print(f"Test accuracy: {grid.score(X_test, y_test):.4f}")
print(f"Time: {grid_time:.1f}s")

## 3. Random Search (same budget)

In [ ]:
param_distributions = {
    "n_estimators": randint(10, 500),
    "max_depth": randint(3, 50),
    "min_samples_split": randint(2, 20),
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions,
    n_iter=27,  # same budget as grid
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
)

start = time.time()
random_search.fit(X_train, y_train)
random_time = time.time() - start

print(f"Best score: {random_search.best_score_:.4f}")
print(f"Best params: {random_search.best_params_}")
print(f"Test accuracy: {random_search.score(X_test, y_test):.4f}")
print(f"Time: {random_time:.1f}s")

### Compare coverage: Grid vs Random

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grid search points
grid_results = grid.cv_results_
ax = axes[0]
sc = ax.scatter(
    grid_results['param_n_estimators'],
    grid_results['param_max_depth'],
    c=grid_results['mean_test_score'],
    cmap='RdYlGn', s=100, edgecolors='black'
)
ax.set_xlabel('n_estimators')
ax.set_ylabel('max_depth')
ax.set_title(f'Grid Search ({len(grid_results["mean_test_score"])} combos)')
plt.colorbar(sc, ax=ax, label='CV Accuracy')

# Random search points
rand_results = random_search.cv_results_
ax = axes[1]
sc = ax.scatter(
    rand_results['param_n_estimators'],
    rand_results['param_max_depth'],
    c=rand_results['mean_test_score'],
    cmap='RdYlGn', s=100, edgecolors='black'
)
ax.set_xlabel('n_estimators')
ax.set_ylabel('max_depth')
ax.set_title(f'Random Search ({len(rand_results["mean_test_score"])} combos)')
plt.colorbar(sc, ax=ax, label='CV Accuracy')

plt.suptitle('Grid vs Random Search: Parameter Coverage', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Count unique values explored
grid_unique_n_est = len(set(grid_results['param_n_estimators']))
rand_unique_n_est = len(set(rand_results['param_n_estimators']))
print(f"Unique n_estimators explored:")
print(f"  Grid:   {grid_unique_n_est} values")
print(f"  Random: {rand_unique_n_est} values")

## 4. Bayesian Optimization with Optuna

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 10, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 50),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
    }
    model = RandomForestClassifier(**params, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
    return scores.mean()

study = optuna.create_study(direction="maximize")

start = time.time()
study.optimize(objective, n_trials=27)  # same budget
optuna_time = time.time() - start

print(f"Best CV score: {study.best_value:.4f}")
print(f"Best params:   {study.best_params}")
print(f"Time: {optuna_time:.1f}s")

# Evaluate on test set
best_model = RandomForestClassifier(**study.best_params, random_state=42)
best_model.fit(X_train, y_train)
print(f"Test accuracy: {best_model.score(X_test, y_test):.4f}")

In [ ]:
# Compare all three methods
print("\n" + "=" * 55)
print(f"{'Method':<20} {'Best CV':>10} {'Test Acc':>10} {'Time':>10}")
print("=" * 55)
print(f"{'Grid Search':<20} {grid.best_score_:>10.4f} {grid.score(X_test, y_test):>10.4f} {grid_time:>9.1f}s")
print(f"{'Random Search':<20} {random_search.best_score_:>10.4f} {random_search.score(X_test, y_test):>10.4f} {random_time:>9.1f}s")
print(f"{'Optuna (Bayesian)':<20} {study.best_value:>10.4f} {best_model.score(X_test, y_test):>10.4f} {optuna_time:>9.1f}s")
print("=" * 55)

### Optuna Visualizations

In [ ]:
# Optimization history
fig = optuna.visualization.plot_optimization_history(study)
fig.show()

In [ ]:
# Parameter importances
fig = optuna.visualization.plot_param_importances(study)
fig.show()

In [ ]:
# Parallel coordinate plot
fig = optuna.visualization.plot_parallel_coordinate(study)
fig.show()

## 5. Cross-Validation Deep Dive

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Standard K-Fold
from sklearn.model_selection import KFold, StratifiedKFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kf = cross_val_score(model, X_train, y_train, cv=kf)

# Stratified K-Fold (preserves class ratio)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_skf = cross_val_score(model, X_train, y_train, cv=skf)

print(f"K-Fold:          {scores_kf.mean():.4f} ± {scores_kf.std():.4f}")
print(f"Stratified K-Fold: {scores_skf.mean():.4f} ± {scores_skf.std():.4f}")
print(f"\nPer-fold scores:")
for i, (s1, s2) in enumerate(zip(scores_kf, scores_skf)):
    print(f"  Fold {i+1}: K-Fold={s1:.4f}, Stratified={s2:.4f}")

## 6. PyTorch Reproducibility

In [ ]:
import torch
import torch.nn as nn
import random
import os

def set_seed_full(seed=42):
    """Complete seed function for PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# Without seeds: different every time
print("=== Without seeds ===")
for i in range(3):
    model = nn.Linear(10, 1)
    x = torch.randn(5, 10)
    y = model(x)
    print(f"  Run {i+1}: first output = {y[0].item():.6f}")

# With seeds: identical every time
print("\n=== With seeds ===")
for i in range(3):
    set_seed_full(42)
    model = nn.Linear(10, 1)
    x = torch.randn(5, 10)
    y = model(x)
    print(f"  Run {i+1}: first output = {y[0].item():.6f}")

In [ ]:
# DataLoader reproducibility
from torch.utils.data import DataLoader, TensorDataset

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print("=== DataLoader with worker seeding ===")
for i in range(3):
    set_seed_full(42)
    g = torch.Generator()
    g.manual_seed(42)
    dataset = TensorDataset(torch.randn(100, 10), torch.randint(0, 2, (100,)))
    loader = DataLoader(
        dataset, batch_size=16, shuffle=True,
        worker_init_fn=seed_worker, generator=g
    )
    first_batch = next(iter(loader))
    print(f"  Run {i+1}: first batch sum = {first_batch[0].sum().item():.4f}, "
          f"labels = {first_batch[1][:5].tolist()}")

### Reporting results over multiple seeds

In [ ]:
# Better than one deterministic run: report mean ± std
results = []
for seed in [42, 123, 456, 789, 1024]:
    np.random.seed(seed)
    X_s = np.random.rand(500, 8)
    y_s = (X_s[:, 0] * 2 + X_s[:, 2] - X_s[:, 5] + np.random.randn(500) * 0.3 > 1).astype(int)
    Xtr, Xte, ytr, yte = train_test_split(X_s, y_s, test_size=0.2, random_state=seed)
    m = RandomForestClassifier(n_estimators=100, random_state=seed)
    m.fit(Xtr, ytr)
    acc = m.score(Xte, yte)
    results.append(acc)
    print(f"  Seed {seed:>4d}: {acc:.4f}")

print(f"\nAccuracy: {np.mean(results):.4f} ± {np.std(results):.4f}")
print("This is more informative than a single deterministic run!")

## 7. W&B Integration (Optional)

Uncomment and run if you have a W&B account.

In [ ]:
# import wandb

# wandb.init(project="movie-predictor-demo", config={
#     "n_estimators": 100,
#     "max_depth": 10,
#     "seed": 42,
# })

# model = RandomForestClassifier(
#     n_estimators=wandb.config.n_estimators,
#     max_depth=wandb.config.max_depth,
#     random_state=wandb.config.seed,
# )
# model.fit(X_train, y_train)

# train_acc = model.score(X_train, y_train)
# test_acc = model.score(X_test, y_test)

# wandb.log({"train_accuracy": train_acc, "test_accuracy": test_acc})

# # Log feature importances
# for i, imp in enumerate(model.feature_importances_):
#     wandb.log({f"feature_{i}_importance": imp})

# print(f"Train: {train_acc:.4f}, Test: {test_acc:.4f}")
# print(f"View at: {wandb.run.url}")
# wandb.finish()

## Summary

| Method | Budget | Best CV Score | Key Advantage |
|--------|--------|--------------|---------------|
| Grid Search | 27 combos | varies | Exhaustive, reproducible |
| Random Search | 27 combos | often better | Explores more unique values |
| Optuna (Bayesian) | 27 trials | often best | Learns from past trials |

**Key takeaways:**
- Random search > grid search for most problems
- Optuna is smarter still (especially with expensive training)
- Always use cross-validation, never tune on the test set
- PyTorch reproducibility requires seeds + deterministic algorithms + DataLoader seeding
- Report results over multiple seeds for reliability